# **Hands-On Pydantic v2 Tutorial: Data Models & Validation**

In this tutorial, we will work through a practical, hands-on tutorial for using **Pydantic v2** to ensure data integrity and robust validation in Python.

**Core Model & Constraints:** Configured standard and constrained fields, including a string name, an automated EmailStr for email format checking, and an age integer field with boundary limits (ge=0).

**Custom Validation Logic:** Built a custom validation rule to ensure that every accountid strictly begins with the required prefix ("ACC-").

**Financial Data Precision:** Integrated Python's Decimal type for monetary attributes like salary to prevent floating-point rounding errors, alongside ISO 4217 Currency validation using pydantic-extra-types.

**Optional Fields:** Handled optional data attributes (such as a phone number) cleanly by combining type hints like Optional[str] with a default value of None.

**Practical Validation & Coercion:** Explored how Pydantic handles real-world inputs by seamlessly coercing data types and raising detailed ValidationError messages when inputs violate constraints.

In [2]:
!pip install pydantic[email] pydantic-extra-types pycountry --q

In [1]:
# BaseModel: Base class to inherit from when creating data models
# EmailStr: Automatically validates proper email address formatting
# Field: Used to set field constraints (like minimum numbers) and metadata
# field_validator: Decorator used to write custom validation logic for a specific field

from pydantic import BaseModel, EmailStr, Field, field_validator

# Import Optional from typing to allow fields to accept a value or None (optional fields)
from typing import Optional

# Import ValidationError to catch and handle errors when data fails validation checks
from pydantic import ValidationError

# Import Currency from pydantic-extra-types to validate standard ISO 4217 currency codes (e.g., "USD")
from pydantic_extra_types.currency_code import Currency

In [2]:
# Define the User data model inheriting from Pydantic's BaseModel
class User(BaseModel):
    # Standard string field for the user's name
    name: str
    
    # EmailStr automatically validates that the input is a valid email address
    email: EmailStr  
    
    # Integer field with a constraint (ge=0 ensures age cannot be negative)
    age: int = Field(ge=0, description="Age must be 0 or older")
    
    # Standard string field for the account identifier
    accountid: str

    # Optional field using Optional[str] and defaulting to None
    phone: Optional[str] = None

    # Standard floating point/decimal field for salary
    salary: float = Field(ge=0.0, description="Salary must be a positive number")

    # Extra Type Currency
    currency: Currency

    # Custom validator to enforce specific business rules on the accountid field
    @field_validator("accountid")
    # In Python, @classmethod is a built-in decorator that tells Python to bind a method to the class 
    # itself rather than to a specific instance of the class.When you validate fields using Pydantic, the 
    # validation happens before the object is actually created.
    @classmethod
    def validate_account_id(cls, value: str) -> str:
        # Check if the account ID begins with the required prefix
        if not value.startswith("ACC-"):
            raise ValueError("Account ID must start with the prefix 'ACC-'")
        return value

### **Valid Data**

In [ ]:
valid_data = {
    "name": "Rahul Gupta",
    "email": "rahulexample.com",
    "age": 30,
    "accountid": "AC-98765",
    "salary": 2000.95,
    "currency": "USD",
    "phone": "91 12345 54321"
}

# Instantiate the model
user = User(**valid_data)
print("Valid User Created Successfully:")
print(user)

ValidationError: 1 validation error for User
accountid
  Value error, Account ID must start with the prefix 'ACC-' [type=value_error, input_value='AC-98765', input_type=str]
    For further information visit https://errors.pydantic.dev/2.8/v/value_error

In [6]:
user.name, user.email, user.age, user.accountid, user.phone, user.salary, user.currency

('Rahul Gupta',
 'rahul@example.com',
 30,
 'ACC-98765',
 '91 12345 54321',
 2000.95,
 'USD')

In [7]:
valid_data = {
    "name": "Rahul Gupta",
    "email": "rahul@example.com",
    "age": '30',
    "accountid": "ACC-98765",
    "salary" : "2000.95",
    "currency" : 'USD'
}

# Instantiate the model
user = User(**valid_data)
print("Valid User Created Successfully:")
print(user)

Valid User Created Successfully:
name='Rahul Gupta' email='rahul@example.com' age=30 accountid='ACC-98765' phone=None salary=2000.95 currency='USD'


In [8]:
user.name, user.email, user.age, user.accountid, user.phone, user.salary, user.currency

('Rahul Gupta', 'rahul@example.com', 30, 'ACC-98765', None, 2000.95, 'USD')

In [9]:
type(user.salary)

float

### **Invalid Data and Error Processing**

In [10]:
invalid_data = {
    "name": "Bob Jones",
    "email": "abcde@xyz",       # Invalid email format
    "age": -5,                    # Violates the ge=0 constraint
    "accountid": "USER-12345",     # Violates the custom validator (doesn't start with 'ACC-'),
    "salary": "abcd",
    "currency": "QAD"
}

try:
    bad_user = User(**invalid_data)
except ValidationError as e:
    print("\nCaught Validation Errors:")
    print(e)


Caught Validation Errors:
5 validation errors for User
email
  value is not a valid email address: The part after the @-sign is not valid. It should have a period. [type=value_error, input_value='abcde@xyz', input_type=str]
age
  Input should be greater than or equal to 0 [type=greater_than_equal, input_value=-5, input_type=int]
    For further information visit https://errors.pydantic.dev/2.8/v/greater_than_equal
accountid
  Value error, Account ID must start with the prefix 'ACC-' [type=value_error, input_value='USER-12345', input_type=str]
    For further information visit https://errors.pydantic.dev/2.8/v/value_error
salary
  Input should be a valid number, unable to parse string as a number [type=float_parsing, input_value='abcd', input_type=str]
    For further information visit https://errors.pydantic.dev/2.8/v/float_parsing
currency
  Invalid currency code. See https://en.wikipedia.org/wiki/ISO_4217 . Bonds, testing and precious metals codes are not allowed. [type=InvalidCurre